In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder


In [3]:
df=pd.read_csv("covid_toy.csv")

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
df.shape

(100, 6)

In [6]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


In [8]:
df.duplicated().sum()

np.int64(1)

In [9]:
df['cough'].value_counts()

,count
cough,
Mild,62
Strong,38


In [10]:
df['city'].value_counts()

,count
city,
Kolkata,32
Bangalore,30
Delhi,22
Mumbai,16


In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X_train,X_test,y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2, random_state=42)

In [13]:
X_train

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi
...,...,...,...,...,...
60,24,Female,102.0,Strong,Bangalore
71,75,Female,104.0,Strong,Delhi
14,51,Male,104.0,Mild,Bangalore
92,82,Female,102.0,Strong,Kolkata


In [14]:
y_train

,has_covid
55,Yes
88,No
26,Yes
42,Yes
69,No
...,...
60,Yes
71,No
14,No
92,No


#Without using Column_Transformer

In [15]:
# addding into the missing value of fever column
si=SimpleImputer()
# using the pbject above
X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.fit_transform(X_test[['fever']])

In [16]:
X_train_fever.shape

(80, 1)

In [26]:
# OneHotEncoding on this Categorical data cough
OE = OneHotEncoder(categories=[['Mild','Strong']], sparse_output=False)
#using the above object for train and test data
X_train_cough = OE.fit_transform(X_train[['cough']])
X_test_cough = OE.fit_transform(X_test[['cough']])

In [34]:
X_train_cough.shape

(80, 2)

In [29]:
#onehotencoding on this categorical data gender and city
ohe = OneHotEncoder(drop='first', sparse_output=False)
#drop = 'first'.. this is done so that we can be safe from multicoliniarity
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

In [35]:
X_train_gender_city.shape

(80, 4)

# Before concatening these new training data we need to seperate the age from the old data

In [32]:
# Extraction of age
# with the training data
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# With the testing data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

In [38]:
X_train_age.shape

(80, 1)

In [41]:
#Now Concatenating all of the new rows and columns
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis = 1)

In [42]:
X_train_transformed.shape

(80, 8)

#**With Column Transformer**

In [44]:
from sklearn.compose import ColumnTransformer

In [45]:
transformer = ColumnTransformer(transformers=[
    ('trf1', SimpleImputer(),['fever']),
    ('trf2', OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']),
    ('trf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender','city'])
], remainder='passthrough')

In [47]:
#Fitting
transformer.fit_transform(X_train)

array([[101.,   0.,   0.,   0.,   0.,   1.,  81.],
       [100.,   0.,   0.,   0.,   1.,   0.,   5.],
       [100.,   0.,   0.,   0.,   1.,   0.,  19.],
       [100.,   0.,   1.,   1.,   0.,   0.,  27.],
       [103.,   0.,   0.,   1.,   0.,   0.,  73.],
       [103.,   1.,   1.,   0.,   1.,   0.,  70.],
       [102.,   0.,   0.,   1.,   0.,   0.,  49.],
       [101.,   1.,   0.,   0.,   1.,   0.,  51.],
       [101.,   0.,   0.,   1.,   0.,   0.,  64.],
       [101.,   0.,   0.,   0.,   1.,   0.,  83.],
       [ 98.,   0.,   0.,   0.,   0.,   1.,  65.],
       [104.,   0.,   0.,   0.,   0.,   0.,  18.],
       [103.,   0.,   0.,   0.,   0.,   0.,  16.],
       [104.,   0.,   1.,   0.,   1.,   0.,  16.],
       [100.,   0.,   1.,   0.,   1.,   0.,  27.],
       [101.,   0.,   0.,   0.,   0.,   0.,  84.],
       [104.,   0.,   1.,   0.,   1.,   0.,  51.],
       [102.,   0.,   0.,   0.,   0.,   0.,  69.],
       [102.,   1.,   0.,   0.,   0.,   0.,  82.],
       [103.,   0.,   0.,   0.,

In [48]:
transformer.fit_transform(X_train).shape

(80, 7)

In [50]:
transformer.transform(X_test).shape

(20, 7)